In [ ]:
import re
import numpy as np

def extract_last_PE(log_path, last_ns=0.1, timestep_fs=1.0):
    """
    Extract the potential energy (PE or PotEng) values from the last 0.1 ns
    of a LAMMPS log file. Works for any thermo style.
    """

    # Convert ns → number of steps
    steps_target = int((last_ns * 1e6) / timestep_fs)

    with open(log_path, "r") as f:
        lines = f.readlines()

    # ---- Find thermo header ----
    header = None
    header_index = None

    for i, line in enumerate(lines):
        if line.startswith("Step") and ("pe" in line or "PotEng" in line):
            header = line.split()
            header_index = i
            break

    if header is None:
        raise RuntimeError("Thermo header not found in log file.")

    # Determine PE column index
    if "pe" in header:
        pe_col = header.index("pe")
    elif "PotEng" in header:
        pe_col = header.index("PotEng")
    else:
        raise RuntimeError("No potential energy column found.")

    step_col = header.index("Step")

    # ---- Extract thermo data ----
    pe_values = []
    for line in lines[header_index+1:]:
        parts = line.split()
        if len(parts) <= pe_col:
            continue
        if parts[step_col].isdigit():
            step = int(parts[step_col])
            pe = float(parts[pe_col])
            pe_values.append((step, pe))

    # ---- Filter last region ----
    max_step = max(s for s,_ in pe_values)
    cutoff_step = max_step - steps_target
    last_pe = [pe for s,pe in pe_values if s >= cutoff_step]

    return np.mean(last_pe)


def extract_last_PE(log_path, thermo_freq=200, last_ns=0.1, timestep_fs=1.0):
    """
    Extract the potential energy (PE) values from the last 0.1 ns of a LAMMPS log file.
    """
    # Convert 0.1 ns → number of steps
    steps_target = int((last_ns * 1e6) / timestep_fs)  # e.g., 0.1 ns → 100000 steps
    
    with open(log_path, "r") as f:
        lines = f.readlines()

    # Find thermo header
    thermo_start = None
    for i, line in enumerate(lines):
        if re.search(r"Step\s+Temp\s+PotEng", line):
            thermo_start = i
            break

    if thermo_start is None:
        raise RuntimeError("Thermo output not found in log file.")

    # Extract thermo data
    pe_values = []
    for line in lines[thermo_start+1:]:
        parts = line.split()
        if len(parts) < 5:  
            continue
        if parts[0].isdigit():  # Step number
            step = int(parts[0])
            pe = float(parts[2])  # 'pe' column
            pe_values.append((step, pe))

    # #save pe_values to file
    # with open("pe_values.txt", "w") as f:
    #     for step, pe in pe_values:
    #         f.write(f"{step}\t{pe}\n")

    # Filter last X steps
    if not pe_values:
        raise RuntimeError("No PE values extracted.")

    max_step = max(s for s, _ in pe_values)
    cutoff_step = max_step - steps_target

    last_pe = [pe for s, pe in pe_values if s >= cutoff_step]
    print(len(last_pe))
    return np.mean(last_pe)


# ---- Compute both ----
close_file_12E = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run1/1_1-12E/close/log.lammps"
away_file_12E  = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run1/1_1-12E/away1/log.lammps"


# ---- Compute both ----
close_file_8E = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run1/2_1-8E/close/log.lammps"
away_file_8E  = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run1/2_1-8E/away1/log.lammps"

# ---- Compute both ----
close_file_6E = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run1/3_1-6E/close/log.lammps"
away_file_6E  = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run1/3_1-6E/away1/log.lammps"


# # ---- Compute both ----
# close_file_12E = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run2/1_1-12E/close/log.lammps"
# away_file_12E  = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run2/1_1-12E/away1/log.lammps"


# # ---- Compute both ----
# close_file_8E = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run2/2_1-8E/close/log.lammps"
# away_file_8E  = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run2/2_1-8E/away1/log.lammps"

# # ---- Compute both ----
# close_file_6E = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run2/3_1-6E/close/log.lammps"
# away_file_6E  = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run2/3_1-6E/away1/log.lammps"


E_close_12E = extract_last_PE(close_file_12E)
E_away_12E  = extract_last_PE(away_file_12E)
E_close_8E  = extract_last_PE(close_file_8E)
E_away_8E   = extract_last_PE(away_file_8E)
E_close_6E  = extract_last_PE(close_file_6E)
E_away_6E   = extract_last_PE(away_file_6E)

E_ads_12E = E_close_12E - E_away_12E
E_ads_8E  = E_close_8E  - E_away_8E
E_ads_6E  = E_close_6E  - E_away_6E


# print("Average PE (close):", E_close)
# print("Average PE (away):", E_away)
print("Adsorption Energy 1-12E (kcal/mol):", E_ads_12E)
print("Adsorption Energy 1-8E  (kcal/mol):", E_ads_8E)
print("Adsorption Energy 1-6E  (kcal/mol):", E_ads_6E)


In [ ]:
import re
import numpy as np

def extract_last_PE(log_path, last_ns=0.1, timestep_fs=1.0):
    """
    Extract the potential energy (PE or PotEng) values from the last 0.1 ns
    of a LAMMPS log file. Works for any thermo style.
    """

    # Convert ns → number of steps
    steps_target = int((last_ns * 1e6) / timestep_fs)

    with open(log_path, "r") as f:
        lines = f.readlines()

    # ---- Find thermo header ----
    header = None
    header_index = None

    for i, line in enumerate(lines):
        if line.startswith("Step") and ("pe" in line or "PotEng" in line):
            header = line.split()
            header_index = i
            break

    if header is None:
        raise RuntimeError("Thermo header not found in log file.")

    # Determine PE column index
    if "pe" in header:
        pe_col = header.index("pe")
    elif "PotEng" in header:
        pe_col = header.index("PotEng")
    else:
        raise RuntimeError("No potential energy column found.")

    step_col = header.index("Step")

    # ---- Extract thermo data ----
    pe_values = []
    for line in lines[header_index+1:]:
        parts = line.split()
        if len(parts) <= pe_col:
            continue
        if parts[step_col].isdigit():
            step = int(parts[step_col])
            pe = float(parts[pe_col])
            pe_values.append((step, pe))

    # ---- Filter last region ----
    max_step = max(s for s,_ in pe_values)
    cutoff_step = max_step - steps_target
    last_pe = [pe for s,pe in pe_values if s >= cutoff_step]

    return np.mean(last_pe)


def extract_last_PE(log_path, thermo_freq=200, last_ns=0.1, timestep_fs=1.0):
    """
    Extract the potential energy (PE) values from the last 0.1 ns of a LAMMPS log file.
    """
    # Convert 0.1 ns → number of steps
    steps_target = int((last_ns * 1e6) / timestep_fs)  # e.g., 0.1 ns → 100000 steps
    
    with open(log_path, "r") as f:
        lines = f.readlines()

    # Find thermo header
    thermo_start = None
    for i, line in enumerate(lines):
        if re.search(r"Step\s+Temp\s+PotEng", line):
            thermo_start = i
            break

    if thermo_start is None:
        raise RuntimeError("Thermo output not found in log file.")

    # Extract thermo data
    pe_values = []
    for line in lines[thermo_start+1:]:
        parts = line.split()
        if len(parts) < 5:  
            continue
        if parts[0].isdigit():  # Step number
            step = int(parts[0])
            pe = float(parts[2])  # 'pe' column
            pe_values.append((step, pe))

    # #save pe_values to file
    # with open("pe_values.txt", "w") as f:
    #     for step, pe in pe_values:
    #         f.write(f"{step}\t{pe}\n")

    # Filter last X steps
    if not pe_values:
        raise RuntimeError("No PE values extracted.")

    max_step = max(s for s, _ in pe_values)
    cutoff_step = max_step - steps_target

    last_pe = [pe for s, pe in pe_values if s >= cutoff_step]
    print(len(last_pe))
    return last_pe, np.mean(last_pe)


# ---- Compute both ----
close_file_12E = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run5/log.lammps"
away_file_12E_1  = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run5/2_peg/log.lammps"
away_file_12E_2  = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run5/3_csh_water/log.lammps"


all_E_close_12E, E_close_12E = extract_last_PE(close_file_12E)
all_E_away_12E_1, E_away_12E_1  = extract_last_PE(away_file_12E_1)
all_E_away_12E_2, E_away_12E_2  = extract_last_PE(away_file_12E_2)

E_ads_12E = E_close_12E - (E_away_12E_1+E_away_12E_2)


print("Average PE (close):", E_close_12E)
print("Average PE (PEG_Water):", E_away_12E_1)
print("Average PE (CSH_Water):", E_away_12E_2)
print("Average PE (away):", E_away_12E_1+E_away_12E_2)
print("Adsorption Energy 1-12E (kcal/mol):", E_ads_12E)

from matplotlib import pyplot as plt
plt.plot(all_E_close_12E, label="Close")
plt.legend()
plt.xlabel("Data Point Index")
plt.ylabel("Potential Energy (kcal/mol)")
plt.title("Potential Energy over Time for Close and Away Configurations")
plt.show()



plt.plot(all_E_away_12E_1, label="Away PEG_Water")
plt.legend()
plt.xlabel("Data Point Index")
plt.ylabel("Potential Energy (kcal/mol)")
plt.title("Potential Energy over Time for Close and Away Configurations")
plt.show()


plt.plot(all_E_away_12E_2, label="Away CSH_Water")
plt.legend()
plt.xlabel("Data Point Index")
plt.ylabel("Potential Energy (kcal/mol)")
plt.title("Potential Energy over Time for Close and Away Configurations")
plt.show()


In [ ]:
import re
import numpy as np

def extract_last_PE(log_path, thermo_freq=200, last_ns=0.1, timestep_fs=1.0):
    """
    Extract the potential energy (PE) values from the last 0.1 ns of a LAMMPS log file.
    """
    # Convert 0.1 ns → number of steps
    steps_target = int((last_ns * 1e6) / timestep_fs)  # e.g., 0.1 ns → 100000 steps
    
    with open(log_path, "r") as f:
        lines = f.readlines()

    # Find thermo header
    thermo_start = None
    for i, line in enumerate(lines):
        if re.search(r"Step\s+Temp\s+PotEng", line):
            thermo_start = i
            break

    if thermo_start is None:
        raise RuntimeError("Thermo output not found in log file.")

    # Extract thermo data
    temp_values = []
    pe_values = []
    te_values = []
    press_values = []
    
    for line in lines[thermo_start+1:]:
        parts = line.split()
        if len(parts) < 5:  
            continue
        if parts[0].isdigit():  # Step number
            step = int(parts[0])
            temp = float(parts[1])  # 'temp' column
            pe = float(parts[2])  # 'pe' column
            te = float(parts[3])  # 'pe' column
            press = float(parts[4])  # 'temp' column
            
            temp_values.append((step, temp))
            pe_values.append((step, pe))
            te_values.append((step, te))
            press_values.append((step, press))

    # #save pe_values to file
    # with open("pe_values.txt", "w") as f:
    #     for step, pe in pe_values:
    #         f.write(f"{step}\t{pe}\n")

    # Filter last X steps
    if not pe_values:
        raise RuntimeError("No PE values extracted.")

    max_step = max(s for s, _ in pe_values)
    cutoff_step = max_step - steps_target
    cutoff_step = 1000
    last_temp = [pe for s, pe in temp_values if s >= (cutoff_step)]
    last_pe = [pe for s, pe in pe_values if s >= (cutoff_step)]
    last_te = [pe for s, pe in te_values if s >= (cutoff_step)]
    last_press = [pe for s, pe in press_values if s >= (cutoff_step)]
    print(len(last_pe))
    return last_temp, last_pe, last_te, last_press, np.mean(last_pe)


# ---- Compute both ----
close_file_12E = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run5/log.lammps"
away_file_12E_1  = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run5/2_peg/log.lammps"
away_file_12E_2  = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run5/3_csh_water/log.lammps"


last_temp_1, last_pe_1, last_te_1, last_press_1, E_close_12E = extract_last_PE(close_file_12E)
last_temp_2, last_pe_2, last_te_2, last_press_2, E_away_12E_1  = extract_last_PE(away_file_12E_1)
last_temp_3, last_pe_3, last_te_3, last_press_3, E_away_12E_2  = extract_last_PE(away_file_12E_2)


E_ads_12E = E_close_12E - (E_away_12E_1+E_away_12E_2)


print("Average PE (close):", E_close_12E)
print("Average PE (PEG_Water):", E_away_12E_1)
print("Average PE (CSH_Water):", E_away_12E_2)
print("Average PE (away):", E_away_12E_1+E_away_12E_2)
print("Adsorption Energy 1-12E (kcal/mol):", E_ads_12E)


from matplotlib import pyplot as plt

# plt.plot(last_pe_1, label="Close")
# plt.legend()
# plt.xlabel("Data Point Index")
# plt.ylabel("Potential Energy (kcal/mol)")
# plt.title("Potential Energy over Time for Close and Away Configurations")
# plt.show()



# plt.plot(last_pe_2, label="Away PEG_Water")
# plt.legend()
# plt.xlabel("Data Point Index")
# plt.ylabel("Potential Energy (kcal/mol)")
# plt.title("Potential Energy over Time for Close and Away Configurations")
# plt.show()


# plt.plot(last_pe_3, label="Away CSH_Water")
# plt.legend()
# plt.xlabel("Data Point Index")
# plt.ylabel("Potential Energy (kcal/mol)")
# plt.title("Potential Energy over Time for Close and Away Configurations")
# plt.show()


# 4*3 plot for temp, pe, te, press for close, away_peg, away_csh
fig, axs = plt.subplots(4, 3, figsize=(15, 10))
axs = axs.flatten()
axs[0].plot(last_temp_1, label="Close Temp")
axs[1].plot(last_temp_2, label="Away PEG_Water Temp")
axs[2].plot(last_temp_3, label="Away CSH_Water Temp")
axs[3].plot(last_pe_1, label="Close PE")
axs[4].plot(last_pe_2, label="Away PEG_Water PE")
axs[5].plot(last_pe_3, label="Away CSH_Water PE")
axs[6].plot(last_te_1, label="Close TE")
axs[7].plot(last_te_2, label="Away PEG_Water TE")
axs[8].plot(last_te_3, label="Away CSH_Water TE")
axs[9].plot(last_press_1, label="Close Press")
axs[10].plot(last_press_2, label="Away PEG_Water Press")
axs[11].plot(last_press_3, label="Away CSH_Water Press")

for ax in axs:
    ax.legend()
    # ax.set_xlabel("Data Point Index")
    # ax.set_ylabel("Value")
    # ax.set_title("Thermo Data over Time")

In [ ]:
E_close_12E = np.mean(all_E_close_12E[5:10])
E_away_12E_1 = np.mean(all_E_away_12E_1[5:10])
E_away_12E_2 = np.mean(all_E_away_12E_2[5:10])

E_ads_12E = E_close_12E - (E_away_12E_1+E_away_12E_2)


print("Average PE (close):", E_close_12E)
print("Average PE (PEG_Water):", E_away_12E_1)
print("Average PE (CSH_Water):", E_away_12E_2)
print("Average PE (away):", E_away_12E_1+E_away_12E_2)
print("Adsorption Energy 1-12E (kcal/mol):", E_ads_12E)

In [ ]:
import re
import numpy as np

def extract_last_PE(log_path, thermo_freq=200, last_ns=0.1, timestep_fs=1.0):
    """
    Extract the potential energy (PE) values from the last 0.1 ns of a LAMMPS log file.
    """
    # Convert 0.1 ns → number of steps
    steps_target = int((last_ns * 1e6) / timestep_fs)  # e.g., 0.1 ns → 100000 steps
    
    with open(log_path, "r") as f:
        lines = f.readlines()

    # Find thermo header
    thermo_start = None
    for i, line in enumerate(lines):
        if re.search(r"Step\s+Temp\s+PotEng", line):
            thermo_start = i
            break

    if thermo_start is None:
        raise RuntimeError("Thermo output not found in log file.")

    # Extract thermo data
    temp_values = []
    pe_values = []
    te_values = []
    press_values = []
    
    for line in lines[thermo_start+1:]:
        parts = line.split()
        if len(parts) < 5:  
            continue
        if parts[0].isdigit():  # Step number
            step = int(parts[0])
            temp = float(parts[1])  # 'temp' column
            pe = float(parts[2])  # 'pe' column
            te = float(parts[3])  # 'pe' column
            press = float(parts[4])  # 'temp' column
            
            temp_values.append((step, temp))
            pe_values.append((step, pe))
            te_values.append((step, te))
            press_values.append((step, press))

    # Filter last X steps
    if not pe_values:
        raise RuntimeError("No PE values extracted.")

    max_step = max(s for s, _ in pe_values)
    cutoff_step = max_step - steps_target
    cutoff_step = 1000
    last_temp = [pe for s, pe in temp_values if s >= (cutoff_step)]
    last_pe = [pe for s, pe in pe_values if s >= (cutoff_step)]
    last_te = [pe for s, pe in te_values if s >= (cutoff_step)]
    last_press = [pe for s, pe in press_values if s >= (cutoff_step)]
    print(len(last_pe))
    return last_temp, last_pe, last_te, last_press, np.mean(last_pe)


# ---- Compute both ----
close_file_12E =   "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run6/1_all3/log.lammps"
away_file_12E_1  = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run6/2_peg/log.lammps"
away_file_12E_2  = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run6/3_csh_water/log.lammps"


last_temp_1, last_pe_1, last_te_1, last_press_1, E_close_12E = extract_last_PE(close_file_12E)
last_temp_2, last_pe_2, last_te_2, last_press_2, E_away_12E_1  = extract_last_PE(away_file_12E_1)
last_temp_3, last_pe_3, last_te_3, last_press_3, E_away_12E_2  = extract_last_PE(away_file_12E_2)


E_ads_12E = E_close_12E - (E_away_12E_1+E_away_12E_2)


print("Average PE (close):", E_close_12E)
print("Average PE (PEG_Water):", E_away_12E_1)
print("Average PE (CSH_Water):", E_away_12E_2)
print("Average PE (away):", E_away_12E_1+E_away_12E_2)
print("Adsorption Energy 1-12E (kcal/mol):", E_ads_12E)


from matplotlib import pyplot as plt

# 4*3 plot for temp, pe, te, press for close, away_peg, away_csh
fig, axs = plt.subplots(4, 3, figsize=(15, 10))
axs = axs.flatten()
axs[0].plot(last_temp_1, label="Close Temp")
axs[1].plot(last_temp_2, label="Away PEG_Water Temp")
axs[2].plot(last_temp_3, label="Away CSH_Water Temp")
axs[3].plot(last_pe_1, label="Close PE")
axs[4].plot(last_pe_2, label="Away PEG_Water PE")
axs[5].plot(last_pe_3, label="Away CSH_Water PE")
axs[6].plot(last_te_1, label="Close TE")
axs[7].plot(last_te_2, label="Away PEG_Water TE")
axs[8].plot(last_te_3, label="Away CSH_Water TE")
axs[9].plot(last_press_1, label="Close Press")
axs[10].plot(last_press_2, label="Away PEG_Water Press")
axs[11].plot(last_press_3, label="Away CSH_Water Press")

for ax in axs:
    ax.legend()


In [ ]:
import re
import numpy as np

def extract_last_PE(log_path, thermo_freq=200, last_ns=0.1, timestep_fs=1.0, cutoff_step=1000000):
    """
    Extract the potential energy (PE) values from the last 0.1 ns of a LAMMPS log file.
    """
    # Convert 0.1 ns → number of steps
    steps_target = int((last_ns * 1e6) / timestep_fs)  # e.g., 0.1 ns → 100000 steps
    
    with open(log_path, "r") as f:
        lines = f.readlines()

    # Find thermo header
    thermo_start = None
    for i, line in enumerate(lines):
        if re.search(r"Step\s+Temp\s+PotEng", line):
            thermo_start = i
            break

    if thermo_start is None:
        raise RuntimeError("Thermo output not found in log file.")

    # Extract thermo data
    temp_values = []
    pe_values = []
    te_values = []
    press_values = []
    
    for line in lines[thermo_start+1:]:
        parts = line.split()
        if len(parts) < 5:  
            continue
        if parts[0].isdigit():  # Step number
            step = int(parts[0])
            temp = float(parts[1])  # 'temp' column
            pe = float(parts[2])  # 'pe' column
            te = float(parts[3])  # 'pe' column
            press = float(parts[4])  # 'temp' column
            
            temp_values.append((step, temp))
            pe_values.append((step, pe))
            te_values.append((step, te))
            press_values.append((step, press))

    # Filter last X steps
    if not pe_values:
        raise RuntimeError("No PE values extracted.")

    max_step = max(s for s, _ in pe_values)
    # cutoff_step = max_step - steps_target
    # cutoff_step = 1000000
                #   2614000
    last_temp = [pe for s, pe in temp_values if s >= (cutoff_step)]
    last_pe = [pe for s, pe in pe_values if s >= (cutoff_step)]
    last_te = [pe for s, pe in te_values if s >= (cutoff_step)]
    last_press = [pe for s, pe in press_values if s >= (cutoff_step)]
    print(len(last_pe))
    return last_temp, last_pe, last_te, last_press, np.mean(last_pe[-1:])


# ---- Compute both ----
close_file_12E =   "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run7/1_all3/log.lammps"
away_file_12E_1  = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run7/2_peg/log.lammps"
away_file_12E_2  = "0_CSH_transfer/CSH_surface/try/2_combined_new/0_run7/3_csh_water/log.lammps"


last_temp_1, last_pe_1, last_te_1, last_press_1, E_close_12E = extract_last_PE(close_file_12E, cutoff_step=10000)
last_temp_2, last_pe_2, last_te_2, last_press_2, E_away_12E_1  = extract_last_PE(away_file_12E_1, cutoff_step=10000)
last_temp_3, last_pe_3, last_te_3, last_press_3, E_away_12E_2  = extract_last_PE(away_file_12E_2, cutoff_step=10000)


E_ads_12E = E_close_12E - (E_away_12E_1+E_away_12E_2)


print("Average PE (close):", E_close_12E)
print("Average PE (PEG_Water):", E_away_12E_1)
print("Average PE (CSH_Water):", E_away_12E_2)
print("Average PE (away):", E_away_12E_1+E_away_12E_2)
print("Adsorption Energy 1-12E (kcal/mol):", E_ads_12E)


from matplotlib import pyplot as plt

# 4*3 plot for temp, pe, te, press for close, away_peg, away_csh
fig, axs = plt.subplots(4, 3, figsize=(15, 10))
axs = axs.flatten()
axs[0].plot(last_temp_1, label="Close Temp")
axs[1].plot(last_temp_2, label="Away PEG_Water Temp")
axs[2].plot(last_temp_3, label="Away CSH_Water Temp")
axs[3].plot(last_pe_1, label="Close PE")
axs[4].plot(last_pe_2, label="Away PEG_Water PE")
axs[5].plot(last_pe_3, label="Away CSH_Water PE")
axs[6].plot(last_te_1, label="Close TE")
axs[7].plot(last_te_2, label="Away PEG_Water TE")
axs[8].plot(last_te_3, label="Away CSH_Water TE")
axs[9].plot(last_press_1, label="Close Press")
axs[10].plot(last_press_2, label="Away PEG_Water Press")
axs[11].plot(last_press_3, label="Away CSH_Water Press")

for ax in axs:
    ax.legend()


In [ ]:
last_temp_1, last_pe_1, last_te_1, last_press_1, E_close_12E = extract_last_PE(close_file_12E, cutoff_step=500000)
last_temp_2, last_pe_2, last_te_2, last_press_2, E_away_12E_1  = extract_last_PE(away_file_12E_1, cutoff_step=500000)
last_temp_3, last_pe_3, last_te_3, last_press_3, E_away_12E_2  = extract_last_PE(away_file_12E_2, cutoff_step=500000)

# 4*3 plot for temp, pe, te, press for close, away_peg, away_csh
fig, axs = plt.subplots(4, 3, figsize=(15, 10))
axs = axs.flatten()
axs[0].plot(last_temp_1, label="Close Temp")
axs[1].plot(last_temp_2, label="Away PEG_Water Temp")
axs[2].plot(last_temp_3, label="Away CSH_Water Temp")
axs[3].plot(last_pe_1, label="Close PE")
axs[4].plot(last_pe_2, label="Away PEG_Water PE")
axs[5].plot(last_pe_3, label="Away CSH_Water PE")
axs[6].plot(last_te_1, label="Close TE")
axs[7].plot(last_te_2, label="Away PEG_Water TE")
axs[8].plot(last_te_3, label="Away CSH_Water TE")
axs[9].plot(last_press_1, label="Close Press")
axs[10].plot(last_press_2, label="Away PEG_Water Press")
axs[11].plot(last_press_3, label="Away CSH_Water Press")

for ax in axs:
    ax.legend()


In [ ]:
last_temp_1, last_pe_1, last_te_1, last_press_1, E_close_12E = extract_last_PE(close_file_12E, cutoff_step=1000000)
last_temp_2, last_pe_2, last_te_2, last_press_2, E_away_12E_1  = extract_last_PE(away_file_12E_1, cutoff_step=1000000)
last_temp_3, last_pe_3, last_te_3, last_press_3, E_away_12E_2  = extract_last_PE(away_file_12E_2, cutoff_step=1000000)

# 4*3 plot for temp, pe, te, press for close, away_peg, away_csh
fig, axs = plt.subplots(4, 3, figsize=(15, 10))
axs = axs.flatten()
axs[0].plot(last_temp_1, label="Close Temp")
axs[1].plot(last_temp_2, label="Away PEG_Water Temp")
axs[2].plot(last_temp_3, label="Away CSH_Water Temp")
axs[3].plot(last_pe_1, label="Close PE")
axs[4].plot(last_pe_2, label="Away PEG_Water PE")
axs[5].plot(last_pe_3, label="Away CSH_Water PE")
axs[6].plot(last_te_1, label="Close TE")
axs[7].plot(last_te_2, label="Away PEG_Water TE")
axs[8].plot(last_te_3, label="Away CSH_Water TE")
axs[9].plot(last_press_1, label="Close Press")
axs[10].plot(last_press_2, label="Away PEG_Water Press")
axs[11].plot(last_press_3, label="Away CSH_Water Press")

for ax in axs:
    ax.legend()


In [ ]:
last_temp_1, last_pe_1, last_te_1, last_press_1, E_close_12E = extract_last_PE(close_file_12E, cutoff_step=2000000)
last_temp_2, last_pe_2, last_te_2, last_press_2, E_away_12E_1  = extract_last_PE(away_file_12E_1, cutoff_step=2000000)
last_temp_3, last_pe_3, last_te_3, last_press_3, E_away_12E_2  = extract_last_PE(away_file_12E_2, cutoff_step=2000000)

# 4*3 plot for temp, pe, te, press for close, away_peg, away_csh
fig, axs = plt.subplots(4, 3, figsize=(15, 10))
axs = axs.flatten()
axs[0].plot(last_temp_1, label="Close Temp")
axs[1].plot(last_temp_2, label="Away PEG_Water Temp")
axs[2].plot(last_temp_3, label="Away CSH_Water Temp")
axs[3].plot(last_pe_1, label="Close PE")
axs[4].plot(last_pe_2, label="Away PEG_Water PE")
axs[5].plot(last_pe_3, label="Away CSH_Water PE")
axs[6].plot(last_te_1, label="Close TE")
axs[7].plot(last_te_2, label="Away PEG_Water TE")
axs[8].plot(last_te_3, label="Away CSH_Water TE")
axs[9].plot(last_press_1, label="Close Press")
axs[10].plot(last_press_2, label="Away PEG_Water Press")
axs[11].plot(last_press_3, label="Away CSH_Water Press")

for ax in axs:
    ax.legend()
